| Attack Technique   | Clean mAP@0.5 | Adversarial mAP@0.5 | Drop      |
|--------------------|---------------|---------------------|-----------|
| FGSM               | 0.927         | 0.312               | -0.615    |
| Fool Detectors     | 0.927         | 0.155               | -0.772    |
| UAP                | 0.927         | 0.443               | -0.484    |
| Disappearance DAG  | 0.935         | 0.547              | -0.388    |
| Targeted DAG       | 0.927         | 0.0947              | -0.8323   |
| PGD 0.02, 0.005    | 0.927         | 0.617               | -0.310   |
| PGD 0.1, 0.025      | 0.927        | 0.215               | -0.712   |

In [1]:
!curl -L "https://universe.roboflow.com/ds/glquRaJDf9?key=1kWinBXVTQ" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip

Streaming output truncated to the last 5000 lines.
  inflating: train/labels/001677_JPG_jpg.rf.414e120831f7e7f30447cb3cb26bdc83.txt  
 extracting: train/labels/001677_JPG_jpg.rf.d0e22a225cbb317066ae2e6efd3dbcc2.txt  
  inflating: train/labels/001678_JPG_jpg.rf.21782caf845bde6574cb51b3f5c74490.txt  
  inflating: train/labels/001678_JPG_jpg.rf.4de99d4836e879bc0a6b13d15eefe346.txt  
  inflating: train/labels/001678_JPG_jpg.rf.d9b987c7dd66459a58b74d90a6970de7.txt  
  inflating: train/labels/001679_JPG_jpg.rf.7f8083798d74d6e40543f776ec116c6a.txt  
  inflating: train/labels/001679_JPG_jpg.rf.c11f3850b8f5630a0b21810e0a256c90.txt  
  inflating: train/labels/001679_JPG_jpg.rf.f0cdb80eb57c2cf67c5306cbf7c19aac.txt  
  inflating: train/labels/001680_JPG_jpg.rf.05690ffb9e45dcb471a95f0b96806331.txt  
  inflating: train/labels/001680_JPG_jpg.rf.8472e670be11dc0438f4a3e001e124bb.txt  
  inflating: train/labels/001680_JPG_jpg.rf.fc6132ab57b715e2192ea08bbc90067d.txt  
  inflating: train/labels/001681_JPG

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [4]:
import os
import torch
import numpy as np
from sklearn.cluster import KMeans
from torchvision import models, transforms
from PIL import Image
from util import sample_images_by_class , preprocess_image
from attacks import fgsm_attack_detector, pgd_attack_detector, targeted_dag_attack
import torch
from ultralytics import YOLO

import sys
import io
class SuppressPrints:  # Context manager to silence function output
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = io.StringIO()
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout = self._original_stdout


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
model_path = "/content/drive/MyDrive/yolo/yolov8n_TrafficSigns.pt"
model = YOLO(model_path)

In [ ]:
import os
import random
from pathlib import Path

def get_image_label_paths(images_dir="/content/train/images", labels_dir="/content/train/labels"):
    """
    Selects image and label paths for adversarial attacks (Targeted DAG, PGD, FGSM, UAP).

    Returns:
        tuple: (dag_paths, pgd_paths, fgsm_paths, uap_paths), each an array of (image_path, label_path) tuples.
    """
    # 1. List all images and verify labels
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    image_files = sorted([f for f in images_dir.glob("*.jpg") if f.is_file()])
    valid_pairs = []
    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            valid_pairs.append((str(img_path), str(label_path)))

    print(f"Found {len(valid_pairs)} valid image-label pairs (expected ~10,590).")

    # 2. Identify images with Green Light (0) or Speed Limit 10 (1)
    weak_class_images = []
    for img_path, label_path in valid_pairs:
        try:
            with open(label_path, "r") as f:
                labels = f.readlines()
                if any(line.strip().startswith(("0 ", "1 ")) for line in labels):
                    weak_class_images.append((img_path, label_path))
        except Exception as e:
            print(f"Error reading {label_path}: {e}")

    print(f"Found {len(weak_class_images)} images with Green Light or Speed Limit 10.")

    # 3. Assign images to Targeted DAG (500: 250 weak + 250 random)
    dag_paths = []
    # Select 250 weak-class images (or all if <250)
    weak_sample = random.sample(weak_class_images, min(250, len(weak_class_images)))
    dag_paths.extend(weak_sample)
    # Select 250 random from remaining (exclude weak sample)
    remaining = [(i, l) for i, l in valid_pairs if (i, l) not in weak_sample]
    dag_paths.extend(random.sample(remaining, 250))

    # 4. Assign remaining images to PGD (2,000)
    remaining = [(i, l) for i, l in remaining if (i, l) not in dag_paths]
    pgd_paths = random.sample(remaining, 2000)

    # 5. Assign remaining to FGSM (1,500)
    remaining = [(i, l) for i, l in remaining if (i, l) not in pgd_paths]
    fgsm_paths = random.sample(remaining, 1500)

    # 6. Assign remaining to UAP (1,500)
    remaining = [(i, l) for i, l in remaining if (i, l) not in fgsm_paths]
    uap_paths = random.sample(remaining, 1500)

    # 7. Verify counts and uniqueness
    print(f"Targeted DAG: {len(dag_paths)} samples")
    print(f"PGD: {len(pgd_paths)} samples")
    print(f"FGSM: {len(fgsm_paths)} samples")
    print(f"UAP: {len(uap_paths)} samples")

    all_paths = dag_paths + pgd_paths + fgsm_paths + uap_paths
    assert len(set((i, l) for i, l in all_paths)) == 5500, "Duplicate paths detected!"

    return dag_paths, pgd_paths, fgsm_paths, uap_paths

# Run the selection
dag_paths, pgd_paths, fgsm_paths, uap_paths = get_image_label_paths()


Found 10590 valid image-label pairs (expected ~10,590).
Found 2224 images with Green Light or Speed Limit 10.
Targeted DAG: 500 samples
PGD: 2000 samples
FGSM: 1500 samples
UAP: 1500 samples


In [ ]:
import os
import torch
from torchvision.utils import save_image
from pathlib import Path
import shutil

# Output directories
output_image_dir = Path("PGD_augmented_images/images")
output_label_dir = Path("PGD_augmented_images/labels")
output_image_dir.mkdir(parents=True, exist_ok=True)
output_label_dir.mkdir(parents=True, exist_ok=True)

i = 0
# Attack loop
for img_path, label_path in pgd_paths:
    img_path = Path(img_path)
    label_path = Path(label_path)

    with SuppressPrints():
        adv_tensor = pgd_attack_detector(
            image_path=str(img_path),
            model_path=model_path,
            epsilon=0.1,  # Smaller for training robustness
            num_steps=10,
            step_size=0.025,
            conf_threshold=0.25,
            random_start=True,
            device="cuda" if torch.cuda.is_available() else "cpu"
            )

    # Skip if attack failed
    if adv_tensor is None:
        continue

    base_name = f"adv_pgd_{img_path.stem}"
    adv_image_path = output_image_dir / f"{base_name}.png"
    adv_label_path = output_label_dir / f"{base_name}.txt"

    save_image(adv_tensor.clone().detach().cpu(), adv_image_path, normalize=True)
    shutil.copy(label_path, adv_label_path)

    i += 1
    if i % 100 == 0:
        print(f"Generated {i} adversarial examples.")


Generated 100 adversarial examples.
Generated 200 adversarial examples.
Generated 300 adversarial examples.
Generated 400 adversarial examples.
Generated 500 adversarial examples.
Generated 600 adversarial examples.
Generated 700 adversarial examples.
Generated 800 adversarial examples.
Generated 900 adversarial examples.
Generated 1000 adversarial examples.
Generated 1100 adversarial examples.
Generated 1200 adversarial examples.
Generated 1300 adversarial examples.
Generated 1400 adversarial examples.
Generated 1500 adversarial examples.
Generated 1600 adversarial examples.
Generated 1700 adversarial examples.
Generated 1800 adversarial examples.
Generated 1900 adversarial examples.


In [ ]:
# prompt: zip PGD_augmented_images

!zip -r /content/PGD_augmented_images.zip /content/PGD_augmented_images


  adding: content/PGD_augmented_images/ (stored 0%)
  adding: content/PGD_augmented_images/labels/ (stored 0%)
  adding: content/PGD_augmented_images/labels/adv_pgd_00003_00027_00026_png.rf.240b0e4ec4b2a617a54b214a1ad6e622.txt (deflated 21%)
  adding: content/PGD_augmented_images/labels/adv_pgd_00008_00038_00019_png.rf.a453fa121b0c7df2bcf2a168ddd0fe66.txt (deflated 19%)
  adding: content/PGD_augmented_images/labels/adv_pgd_FisheyeCamera_1_00033_png.rf.6d65bd3913e93339bc15cbcdcec99ce3.txt (deflated 35%)
  adding: content/PGD_augmented_images/labels/adv_pgd_00002_00000_00026_png.rf.d46f9f6501359b9756c23e2262b3197e.txt (deflated 26%)
  adding: content/PGD_augmented_images/labels/adv_pgd_000847_jpg.rf.c628841b5736394b913c505851c45f05.txt (deflated 15%)
  adding: content/PGD_augmented_images/labels/adv_pgd_FisheyeCamera_1_00177_png.rf.4fd6ef4933c8ef258c42c8483dc6d9d7.txt (deflated 21%)
  adding: content/PGD_augmented_images/labels/adv_pgd_FisheyeCamera_1_00069_png.rf.899e0cd56a172c058802663

In [ ]:
!mv PGD_augmented_images.zip /content/drive/MyDrive/yolo

In [ ]:
!ls /content/drive/MyDrive/yolo

adversarial_data_set_dis_dag.zip	 adversarial_dataset_targeted_dag.zip
adversarial_data_set_fgsm.zip		 adversarial_data_set_UAP.zip
adversarial_data_set_fool_detectors.zip  fgsm_augmented_images.zip
adversarial_data_set_pgd_02.zip		 PGD_augmented_images.zip
adversarial_data_set_pgd_1.zip		 yolov8n_TrafficSigns.pt


In [ ]:
import os
import torch
from torchvision.utils import save_image
from pathlib import Path
import shutil

# Output directories
output_image_dir = Path("fgsm_augmented_images/images")
output_label_dir = Path("fgsm_augmented_images/labels")
output_image_dir.mkdir(parents=True, exist_ok=True)
output_label_dir.mkdir(parents=True, exist_ok=True)

i = 0
# Attack loop
for img_path, label_path in fgsm_paths:
    img_path = Path(img_path)
    label_path = Path(label_path)

    with SuppressPrints():
        adv_tensor = fgsm_attack_detector(
            image_path=str(img_path),
            model_path=model_path,
            epsilon=0.15,
            conf_threshold=0.25,
            device="cuda" if torch.cuda.is_available() else "cpu"
        )

    # Skip if attack failed
    if adv_tensor is None:
        continue

    base_name = f"adv_fgsm_{img_path.stem}"
    adv_image_path = output_image_dir / f"{base_name}.png"
    adv_label_path = output_label_dir / f"{base_name}.txt"

    save_image(adv_tensor.clone().detach().cpu(), adv_image_path, normalize=True)
    shutil.copy(label_path, adv_label_path)

    i += 1
    if i % 100 == 0:
        print(f"Generated {i} adversarial examples.")


Generated 100 adversarial examples.
Generated 200 adversarial examples.
Generated 300 adversarial examples.
Generated 400 adversarial examples.
Generated 500 adversarial examples.
Generated 600 adversarial examples.
Generated 700 adversarial examples.
Generated 800 adversarial examples.
Generated 900 adversarial examples.
Generated 1000 adversarial examples.
Generated 1100 adversarial examples.
Generated 1200 adversarial examples.
Generated 1300 adversarial examples.
Generated 1400 adversarial examples.


In [ ]:
!zip -r /content/fgsm_augmented_images.zip /content/fgsm_augmented_images

!mv fgsm_augmented_images.zip /content/drive/MyDrive/yolo


  adding: content/fgsm_augmented_images/ (stored 0%)
  adding: content/fgsm_augmented_images/labels/ (stored 0%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_000565_jpg.rf.908fd7d48a31ba1b8bab45838b8b98c1.txt (deflated 15%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_00002_00031_00027_png.rf.1487d6446c4d3d823ed1231bffd3f379.txt (deflated 24%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_000480_jpg.rf.029085b0c7e0b472a6d3accbdb595884.txt (deflated 31%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_road352_png.rf.9af0202482d2af127fed33305e693e71.txt (deflated 13%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_road482_png.rf.ab577b8450c69f9d8cda37908dd216b9.txt (deflated 13%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_00002_00015_00004_png.rf.bfac0c8d374b36dc75141baae164a658.txt (deflated 20%)
  adding: content/fgsm_augmented_images/labels/adv_fgsm_000949_jpg.rf.c8b693b2ade3d9a716babda9890f18a2.txt (deflated 37%)
  adding:

In [ ]:
delta_1_epsilon = torch.load("universal_delta_1_epsilon.pth")

In [ ]:
import os
import torch
from torchvision.utils import save_image
from pathlib import Path
import shutil

# Output directories
output_image_dir = Path("uap_augmented_images/images")
output_label_dir = Path("uap_augmented_images/labels")
output_image_dir.mkdir(parents=True, exist_ok=True)
output_label_dir.mkdir(parents=True, exist_ok=True)
delta_1_epsilon = delta_1_epsilon.to("cuda")

i = 0
# Attack loop
for img_path, label_path in uap_paths:
    img_path = Path(img_path)
    label_path = Path(label_path)

    test_image = preprocess_image(img_path).to("cuda")

    # Add the adversarial perturbation
    adv_tensor = test_image + delta_1_epsilon

    # Ensure the adversarial image is in a valid range (e.g., [0, 1])
    adv_tensor = torch.clamp(adv_tensor, 0, 1)

    # Skip if attack failed
    if adv_tensor is None:
        continue

    base_name = f"adv_uap_{img_path.stem}"
    adv_image_path = output_image_dir / f"{base_name}.png"
    adv_label_path = output_label_dir / f"{base_name}.txt"

    save_image(adv_tensor.clone().detach().cpu(), adv_image_path, normalize=True)
    shutil.copy(label_path, adv_label_path)

    i += 1
    if i % 100 == 0:
        print(f"Generated {i} adversarial examples.")


Generated 100 adversarial examples.
Generated 200 adversarial examples.
Generated 300 adversarial examples.
Generated 400 adversarial examples.
Generated 500 adversarial examples.
Generated 600 adversarial examples.
Generated 700 adversarial examples.
Generated 800 adversarial examples.
Generated 900 adversarial examples.
Generated 1000 adversarial examples.
Generated 1100 adversarial examples.
Generated 1200 adversarial examples.
Generated 1300 adversarial examples.
Generated 1400 adversarial examples.
Generated 1500 adversarial examples.


In [ ]:
!zip -r /content/uap_augmented_images.zip /content/uap_augmented_images
!mv uap_augmented_images.zip /content/drive/MyDrive/yolo
!ls /content/drive/MyDrive/yolo


  adding: content/uap_augmented_images/ (stored 0%)
  adding: content/uap_augmented_images/labels/ (stored 0%)
  adding: content/uap_augmented_images/labels/adv_uap_00000_00000_00011_png.rf.d90e4f357e6863b504552c29425e5a41.txt (deflated 18%)
  adding: content/uap_augmented_images/labels/adv_uap_00003_00000_00019_png.rf.9be039400ecf2ee302461849eb11c06e.txt (deflated 24%)
  adding: content/uap_augmented_images/labels/adv_uap_00001_00060_00000_png.rf.f147190985006ebe0a4886b529f9bd50.txt (deflated 23%)
  adding: content/uap_augmented_images/labels/adv_uap_000950_JPG_jpg.rf.ed045115e468d3a4f63990a2c7617745.txt (deflated 14%)
  adding: content/uap_augmented_images/labels/adv_uap_00000_00006_00029_png.rf.b4ad2ff118d9c1470180a74dbd7a662a.txt (deflated 19%)
  adding: content/uap_augmented_images/labels/adv_uap_road528_png.rf.318f9cf9f23f5e6b30faab08fd16fd0b.txt (deflated 28%)
  adding: content/uap_augmented_images/labels/adv_uap_FisheyeCamera_1_00362_png.rf.4f8f57c44b60eedbc040b81b67ab6238.txt 

In [ ]:
from operator import mod
import os
import torch
from torchvision.utils import save_image
from pathlib import Path
import shutil

# Output directories
output_image_dir = Path("Tdag_augmented_images/images")
output_label_dir = Path("Tdag_augmented_images/labels")
output_image_dir.mkdir(parents=True, exist_ok=True)
output_label_dir.mkdir(parents=True, exist_ok=True)

i = 0
for img_path, label_path in dag_paths:
    # Get original detections
    results = model(img_path)
    detected_classes = results[0].boxes.cls.int().tolist()

    # Determine target class
    adversarial_class = 1 if 0 in detected_classes else 0

    # Run attack with dynamic target
    with SuppressPrints():
        adv_tensor, success = targeted_dag_attack(
            image_path=img_path,
            model_path=model_path,
            num_iterations=20,
            gamma=0.003,
            adversarial_class=adversarial_class,
            conf_threshold=0.25,
            device="cuda" if torch.cuda.is_available() else "cpu"
        )

    # Skip if attack failed
    if adv_tensor is None:
        continue

    # Fixed line: Convert img_path to Path object to access stem
    base_name = f"adv_dag_{Path(img_path).stem}"
    adv_image_path = output_image_dir / f"{base_name}.png"
    adv_label_path = output_label_dir / f"{base_name}.txt"

    save_image(adv_tensor.clone().detach().cpu(), adv_image_path, normalize=True)
    shutil.copy(label_path, adv_label_path)

    i += 1
    if i % 100 == 0:
        print(f"Generated {i} adversarial examples.")


Streaming output truncated to the last 5000 lines.

0: 640x640 1 Speed Limit 80, 10.1ms
Speed: 0.0ms preprocess, 10.1ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 80, 11.7ms
Speed: 0.1ms preprocess, 11.7ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 80, 11.1ms
Speed: 0.0ms preprocess, 11.1ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 80, 10.1ms
Speed: 0.0ms preprocess, 10.1ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 10.5ms
Speed: 0.0ms preprocess, 10.5ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 80, 11.7ms
Speed: 0.0ms preprocess, 11.7ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 80, 13.5ms
Speed: 0.0ms preprocess, 13.5ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 

In [ ]:
# prompt: zip Tdag_augmented_images

!zip -r /content/Tdag_augmented_images.zip /content/Tdag_augmented_images
!mv Tdag_augmented_images.zip /content/drive/MyDrive/yolo
!ls /content/drive/MyDrive/yolo


  adding: content/Tdag_augmented_images/ (stored 0%)
  adding: content/Tdag_augmented_images/labels/ (stored 0%)
  adding: content/Tdag_augmented_images/labels/adv_dag_000817_jpg.rf.68f00bacd64044c26863288fe30662fc.txt (deflated 55%)
  adding: content/Tdag_augmented_images/labels/adv_dag_000876_jpg.rf.f1484438b786c9bc2472eb5256de0105.txt (deflated 37%)
  adding: content/Tdag_augmented_images/labels/adv_dag_001628_JPG_jpg.rf.d9bd8fdce7633b12b74945d83f2ac9ae.txt (deflated 21%)
  adding: content/Tdag_augmented_images/labels/adv_dag_road464_png.rf.4271b0c2de0fab3e46a1b9a33186641e.txt (deflated 34%)
  adding: content/Tdag_augmented_images/labels/adv_dag_00001_00036_00016_png.rf.2e598c49c4d3a6bf8537f5b2fc81dacb.txt (deflated 16%)
  adding: content/Tdag_augmented_images/labels/adv_dag_000962_jpg.rf.b13038a44582de579d9ebd346baca2dd.txt (deflated 37%)
  adding: content/Tdag_augmented_images/labels/adv_dag_road474_png.rf.ad3e138e4e43508bccf7c4a88b3f0db6.txt (deflated 21%)
  adding: content/Tdag_

In [ ]:
!ls /content/drive/MyDrive/yolo/uap_augmented_images.zip

adversarial_data_set_dis_dag.zip	 adversarial_data_set_UAP.zip
adversarial_data_set_fgsm.zip		 fgsm_augmented_images.zip
adversarial_data_set_fool_detectors.zip  PGD_augmented_images.zip
adversarial_data_set_pgd_02.zip		 Tdag_augmented_images.zip
adversarial_data_set_pgd_1.zip		 uap_augmented_images.zip
adversarial_dataset_targeted_dag.zip	 yolov8n_TrafficSigns.pt


In [7]:
# prompt: copy PGD_augmented_images.zip and fgsm_augmented_images.zip and Tdag_augmented_images.zip uap_augmented_images.zip to /content/

!cp /content/drive/MyDrive/yolo/PGD_augmented_images.zip /content/
!unzip PGD_augmented_images.zip
!mv /content/content/PGD_augmented_images /content/PGD_augmented_images
!rm -rf /content/content/


!cp /content/drive/MyDrive/yolo/Tdag_augmented_images.zip /content/
!unzip /content/Tdag_augmented_images.zip
!mv /content/content/Tdag_augmented_images /content/Tdag_augmented_images
!rm -rf /content/content


!cp /content/drive/MyDrive/yolo/fgsm_augmented_images.zip /content/
!unzip fgsm_augmented_images.zip
!mv /content/content/fgsm_augmented_images /content/fgsm_augmented_images
!rm -rf /content/content/


!cp /content/drive/MyDrive/yolo/uap_augmented_images.zip /content/
!unzip uap_augmented_images.zip
!mv /content/content/uap_augmented_images /content/uap_augmented_images
!rm -rf /content/content/

!cp /content/drive/MyDrive/yolo/fgsm2_augmented_images.zip /content/
!unzip fgsm2_augmented_images.zip
!mv /content/content/fgsm2_augmented_images /content/fgsm2_augmented_images
!rm -rf /content/content/


!cp /content/drive/MyDrive/yolo/fool_augmented_images.zip /content/
!unzip fool_augmented_images.zip
!mv /content/content/fool_augmented_images /content/fool_augmented_images
!rm -rf /content/content/

Streaming output truncated to the last 5000 lines.
  inflating: fgsm2_augmented_images/images/adv_fgsm_00001_00064_00011_png.rf.fe82301549b1a458dd9e56b814a72694.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_001013_jpg.rf.92a05273efd58371e5ec74cd45704a6d.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_02911acb-1e9f-4dca-a5fa-22aec97e04aa_jpg.rf.a56c1510b1112511648882c727785e70.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_00000_00001_00023_png.rf.a9f752c59ce5a8f03b49589194dcfff8.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_FisheyeCamera_1_00175_png.rf.10940ff762aeceb2c51d464fe1d36387.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_00003_00038_00027_png.rf.5e84563a1733d2013045d15e6ba27fe6.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_00003_00044_00018_png.rf.0cf278bfc96b9fac9ec9633b71d782da.png  
  inflating: fgsm2_augmented_images/images/adv_fgsm_FisheyeCamera_1_00109_png.rf.b0cd361c355011be2590c40c436a4b65.png  
  inf

In [11]:
!mkdir /content/fool_augmented_images/

In [12]:
!mv /content/images/ /content/fool_augmented_images/
!mv /content/labels/ /content/fool_augmented_images/

In [13]:
# prompt: merge content/PGD_augmented_images/images and content/fgsm_augmented_images/images and content/Tdag_augmented_images/images and content/uap_augmented_images/images images into /content/train/images

import shutil
from pathlib import Path

# Define source and destination directories
source_dirs = [
    "/content/PGD_augmented_images/images",
    "/content/fgsm_augmented_images/images",
    "/content/Tdag_augmented_images/images",
    "/content/uap_augmented_images/images",
    "/content/fgsm2_augmented_images/images",
    "/content/fool_augmented_images/images"
]
destination_dir = "/content/train/images"

# Create the destination directory if it doesn't exist
Path(destination_dir).mkdir(parents=True, exist_ok=True)

# Iterate through source directories and copy images
for source_dir in source_dirs:
    for filename in Path(source_dir).glob("*"):
        if filename.is_file():
            shutil.copy2(filename, destination_dir) # copy2 preserves metadata


In [14]:
# prompt: how many files in /content/train/images

import os

image_dir = "/content/train/images"
num_files = len([f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))])
print(f"Number of files in {image_dir}: {num_files}")


Number of files in /content/train/images: 18842


In [15]:
# prompt: merge content/PGD_augmented_images/labels and content/fgsm_augmented_images/labels and content/Tdag_augmented_images/labels and content/uap_augmented_images/labels images into /content/train/labels

# Define source and destination directories for labels
source_dirs = [
    "/content/PGD_augmented_images/labels",
    "/content/fgsm_augmented_images/labels",
    "/content/Tdag_augmented_images/labels",
    "/content/uap_augmented_images/labels",
    "/content/fgsm2_augmented_images/labels",
    "/content/fool_augmented_images/labels"
]
destination_dir = "/content/train/labels"

# Create the destination directory if it doesn't exist
Path(destination_dir).mkdir(parents=True, exist_ok=True)

# Iterate through source directories and copy labels
for source_dir in source_dirs:
    for filename in Path(source_dir).glob("*"):
        if filename.is_file():
            shutil.copy2(filename, destination_dir) # copy2 preserves metadata

#Verify number of labels
label_dir = "/content/train/labels"
num_files = len([f for f in os.listdir(label_dir) if os.path.isfile(os.path.join(label_dir, f))])
print(f"Number of files in {label_dir}: {num_files}")


Number of files in /content/train/labels: 18842


In [16]:
model = YOLO(model_path)


result = model.train(data='/content/data.yaml', epochs=10, batch=32)


Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/yolo/yolov8n_TrafficSigns.pt, data=/content/data.yaml, epochs=10, time=None, patience=100, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=T

100%|██████████| 755k/755k [00:00<00:00, 14.1MB/s]


                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128

 22        [15, 18, 21]  1    754237  ultralytics.nn.modules.head.Detect           [15, [64, 128, 256]]          
Model summary: 129 layers, 3,013,773 parameters, 3,013,757 gradients, 8.2 GFLOPs

Transferred 70/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


100%|██████████| 5.35M/5.35M [00:00<00:00, 66.4MB/s]


...

      10/10      5.12G     0.5286     0.3615     0.9503         28        640: 100%|██████████| 589/589 [06:38<00:00,  1.48it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:07<00:00,  1.75it/s]

                   all        801        944      0.926      0.917      0.957       0.82



10 epochs completed in 1.139 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train/weights/best.pt, 6.2MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:13<00:00,  1.04s/it]


                   all        801        944      0.942      0.904      0.957      0.821
           Green Light         87        122      0.886        0.7      0.844      0.515
             Red Light         74        108      0.882      0.691      0.826      0.479
       Speed Limit 100         52         52      0.878      0.981      0.986      0.884
       Speed Limit 110         17         17      0.849      0.882       0.95      0.884
       Speed Limit 120         60         60      0.998      0.967      0.995      0.913
        Speed Limit 20         56         56      0.987      0.982      0.986      0.879
        Speed Limit 30         71         74       0.96      0.959       0.99      0.893
        Speed Limit 40         53         55      0.931      0.976      0.992      0.865
        Speed Limit 50         68         71      0.959      0.873      0.959      0.845
        Speed Limit 60         76         76      0.939      0.934       0.96      0.872
        Speed Limit 7

In [17]:
# Save the trained model to Google Drive
model_save_path = '/content/drive/MyDrive/yolo/yolov8n_TrafficSigns_adversarial_training2.pt'
model.export(format='onnx', imgsz=640, dynamic=True)
model.save(model_save_path)
print(f"Model saved to: {model_save_path}")

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.30GHz)
Model summary (fused): 72 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 19, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0', 'onnxslim>=0.1.46', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 161.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 241.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.8/280.8 MB 272.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 269.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 322.0 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 16.3s, installed 3 packages: ['onnx>=1.12.0', 'onnxslim>=0.1.46', 'onnxruntime-gpu']
WARNING ⚠️ requirements: Restart runtime or 

In [18]:
# Evaluate on the test set
results = model.val(
    data="/content/data.yaml",
    split="test",  # Explicitly specify the test split
    batch=32,      # Adjust based on GPU memory
    imgsz=640,      # Match training image size
    conf= 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 9.5±4.0 MB/s, size: 37.7 KB)


val: Scanning /content/test/labels... 638 images, 1 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<00:00, 857.50it/s]

val: New cache created: /content/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:13<00:00,  1.48it/s]


                   all        638        770      0.939      0.877      0.941       0.82
           Green Light         77        110      0.918      0.713      0.897      0.589
             Red Light         71         94      0.914      0.691      0.832      0.538
        Speed Limit 10          2          3      0.765      0.667      0.749      0.674
       Speed Limit 100         45         46      0.914      0.925      0.954      0.856
       Speed Limit 110         21         21      0.964       0.81       0.96      0.851
       Speed Limit 120         40         44        0.9      0.955      0.982      0.878
        Speed Limit 20         46         46      0.957      0.973      0.986      0.911
        Speed Limit 30         60         60      0.963      0.872      0.951      0.894
        Speed Limit 40         51         53       0.97      0.962       0.98      0.866
        Speed Limit 50         47         50      0.977       0.92      0.961      0.878
        Speed Limit 6

In [19]:
!cp /content/drive/MyDrive/yolo/adversarial_data_set_pgd_1.zip /content/


In [20]:

!unzip adversarial_data_set_pgd_1.zip -d /content/


Archive:  adversarial_data_set_pgd_1.zip
   creating: /content/content/adversarial_dataset/
  inflating: /content/content/adversarial_dataset/labels.cache  
   creating: /content/content/adversarial_dataset/labels/
  inflating: /content/content/adversarial_dataset/labels/00004_00060_00021_png.rf.a81a5ba2fb654d944063ddd496a46662.txt  
  inflating: /content/content/adversarial_dataset/labels/road446_png.rf.c209c8b5929313d1258667af1a6a37ae.txt  
  inflating: /content/content/adversarial_dataset/labels/00001_00028_00017_png.rf.98d01270a314b8cc77f6ad92f979b47f.txt  
  inflating: /content/content/adversarial_dataset/labels/001261_jpg.rf.7aab90697a4784411cbc2785e626e004.txt  
  inflating: /content/content/adversarial_dataset/labels/road59_png.rf.40b97d5dddbee9679979779c88bb7348.txt  
  inflating: /content/content/adversarial_dataset/labels/000811_jpg.rf.f5bb7753902ae5b5fd8f5e7f601da271.txt  
  inflating: /content/content/adversarial_dataset/labels/road692_png.rf.ab199ab022b948479bfba7b3c8fd66

In [21]:
# Evaluate pgd with 0.1 epsilon
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3314.5±1455.6 MB/s, size: 961.6 KB)


val: Scanning /content/content/adversarial_dataset/labels... 636 images, 1 backgrounds, 0 corrupt: 100%|██████████| 636/636 [00:01<00:00, 616.87it/s]


val: New cache created: /content/content/adversarial_dataset/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:14<00:00,  1.34it/s]


                   all        636        768      0.899      0.868       0.92      0.795
           Green Light         77        110      0.976      0.726      0.866      0.554
             Red Light         71         94      0.839       0.67       0.78      0.509
        Speed Limit 10          2          3      0.667      0.667      0.749      0.674
       Speed Limit 100         45         46      0.826      0.935      0.961      0.854
       Speed Limit 110         21         21      0.902      0.875      0.918      0.787
       Speed Limit 120         40         44      0.855      0.932       0.96      0.847
        Speed Limit 20         46         46      0.923      0.978      0.986      0.907
        Speed Limit 30         60         60      0.889       0.85      0.909      0.845
        Speed Limit 40         51         53      0.944      0.962      0.978      0.852
        Speed Limit 50         46         49      0.947      0.878       0.94      0.856
        Speed Limit 6

In [23]:
!rm -rf /content/content/
!cp /content/drive/MyDrive/yolo/adversarial_data_set_fgsm.zip /content/

!unzip adversarial_data_set_fgsm.zip -d /content/


Archive:  adversarial_data_set_fgsm.zip
   creating: /content/content/adversarial_dataset/
   creating: /content/content/adversarial_dataset/labels/
  inflating: /content/content/adversarial_dataset/labels/road466_png.rf.e1995d1db602b4fe4f26c9a2f882bf06.txt  
  inflating: /content/content/adversarial_dataset/labels/road654_png.rf.46b39454d20cf1f488267148cc1ee23c.txt  
  inflating: /content/content/adversarial_dataset/labels/road314_png.rf.b9b5f3356645589e1d35652eaf64b5bf.txt  
  inflating: /content/content/adversarial_dataset/labels/00005_00016_00001_png.rf.8ed5c07e98a19127d55583cbd0ebabfb.txt  
  inflating: /content/content/adversarial_dataset/labels/00002_00032_00021_png.rf.da1c231b696e926dff55375e5718a461.txt  
  inflating: /content/content/adversarial_dataset/labels/road108_png.rf.3ccb5b5dbd4fe2ff044ce6f2aed3cf40.txt  
  inflating: /content/content/adversarial_dataset/labels/643854b9-b9d9-4b74-99fb-830fde4a0b74_jpg.rf.70299bd9ade4fba6166556c81ee6a4c0.txt  
  inflating: /content/con

In [24]:
!mv /content/content/adversarial_dataset/ /content/adversarial_dataset
!rm -rf /content/content/

In [25]:
# Evaluate fgsm
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1166.7±606.7 MB/s, size: 582.7 KB)


val: Scanning /content/adversarial_dataset/labels.cache... 638 images, 1 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:19<00:00,  1.03it/s]


                   all        638        770      0.921      0.769      0.874      0.753
           Green Light         77        110      0.931      0.494      0.737       0.48
             Red Light         71         94      0.872      0.436      0.652      0.421
        Speed Limit 10          2          3      0.986          1      0.995      0.843
       Speed Limit 100         45         46      0.813      0.848      0.875      0.773
       Speed Limit 110         21         21      0.873      0.619      0.816      0.747
       Speed Limit 120         40         44       0.84      0.886      0.902       0.77
        Speed Limit 20         46         46      0.934      0.957      0.974      0.872
        Speed Limit 30         60         60       0.94       0.78      0.891      0.831
        Speed Limit 40         51         53       0.94       0.88      0.938      0.811
        Speed Limit 50         47         50      0.949      0.738      0.854      0.765
        Speed Limit 6

In [26]:
!rm -rf /content/adversarial_dataset/
!cp /content/drive/MyDrive/yolo/adversarial_dataset_targeted_dag.zip /content/
!unzip adversarial_dataset_targeted_dag.zip -d /content/
!mv /content/content/adversarial_dataset/ /content/adversarial_dataset
!rm -rf /content/content/

Archive:  adversarial_dataset_targeted_dag.zip
   creating: /content/adversarial_dataset_targeted_dag/
   creating: /content/adversarial_dataset_targeted_dag/content/
   creating: /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/
  inflating: /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/labels.cache  
   creating: /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/images/
  inflating: /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/images/road321_png.rf.273e027200cfc85d8f8084d63ca95d07.png  
  inflating: /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/images/road708_png.rf.4770835b7ec7f0e5c66a8a9ec37d5ee2.png  
  inflating: /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/images/000626_jpg.rf.b953e9072596ff4b4b28334c314e2d56.png  
  inflating: /content/adversarial_dataset_targeted_dag/content/adversarial_

In [27]:
!mv /content/adversarial_dataset_targeted_dag/content/adversarial_dataset_targeted/ /content/adversarial_dataset/

In [28]:
!rm -rf /content/adversarial_dataset_targeted_dag

In [29]:
# Evaluate targeted dag
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1927.4±974.3 MB/s, size: 659.0 KB)


val: Scanning /content/adversarial_dataset/labels... 638 images, 1 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<00:00, 916.59it/s]

val: New cache created: /content/adversarial_dataset/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:15<00:00,  1.33it/s]


                   all        638        770      0.931      0.855      0.927      0.809
           Green Light         77        110      0.961      0.675      0.869      0.579
             Red Light         71         94      0.861      0.593      0.805      0.518
        Speed Limit 10          2          3      0.819      0.667      0.749      0.674
       Speed Limit 100         45         46      0.899      0.913       0.95      0.842
       Speed Limit 110         21         21      0.917      0.857      0.928      0.827
       Speed Limit 120         40         44      0.867      0.932      0.969      0.875
        Speed Limit 20         46         46      0.937      0.965      0.986      0.908
        Speed Limit 30         60         60      0.929      0.869      0.943      0.894
        Speed Limit 40         51         53      0.981      0.954      0.979      0.865
        Speed Limit 50         47         50      0.936      0.872       0.95       0.86
        Speed Limit 6

In [30]:
!rm -rf /content/adversarial_dataset/
!cp /content/drive/MyDrive/yolo/adversarial_data_set_UAP.zip /content/
!unzip adversarial_data_set_UAP.zip -d /content/
!mv /content/content/adversarial_dataset/ /content/adversarial_dataset

Archive:  adversarial_data_set_UAP.zip
   creating: /content/content/adversarial_dataset/
  inflating: /content/content/adversarial_dataset/labels.cache  
   creating: /content/content/adversarial_dataset/images/
  inflating: /content/content/adversarial_dataset/images/road619_png.rf.ec0349b1b45cd38c36b9110291602e31.png  
  inflating: /content/content/adversarial_dataset/images/00002_00000_00016_png.rf.0272cba0fffe74ca2a40dccffdd33270.png  
  inflating: /content/content/adversarial_dataset/images/00001_00033_00010_png.rf.70c439be8b5a8ac7e9b7ca4ba967f18b.png  
  inflating: /content/content/adversarial_dataset/images/000006_jpg.rf.b85f0712823e684de9e077648805dd9c.png  
  inflating: /content/content/adversarial_dataset/images/road51_png.rf.9cba33624ddf052b3dd78e7610764af0.png  
  inflating: /content/content/adversarial_dataset/images/road650_png.rf.0dd9a992c68d930e3100278e1558560f.png  
  inflating: /content/content/adversarial_dataset/images/00014_00011_00027_png.rf.14cd8d067b76ef9c97afe

In [31]:
# Evaluate uap
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2670.0±1704.4 MB/s, size: 929.4 KB)


val: Scanning /content/adversarial_dataset/labels.cache... 638 images, 1 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:16<00:00,  1.18it/s]


                   all        638        770      0.931      0.835      0.917      0.802
           Green Light         77        110      0.946      0.632      0.829      0.559
             Red Light         71         94      0.922      0.628      0.815      0.538
        Speed Limit 10          2          3      0.811      0.667      0.749      0.674
       Speed Limit 100         45         46      0.887      0.913      0.952      0.852
       Speed Limit 110         21         21      0.943      0.795      0.892      0.811
       Speed Limit 120         40         44      0.892      0.944      0.979      0.876
        Speed Limit 20         46         46      0.947      0.957      0.984      0.901
        Speed Limit 30         60         60      0.944      0.845      0.922      0.872
        Speed Limit 40         51         53      0.963      0.973      0.989      0.858
        Speed Limit 50         47         50      0.953      0.817      0.927      0.845
        Speed Limit 6

In [32]:
!rm -rf /content/adversarial_dataset/

In [33]:
!cp /content/drive/MyDrive/yolo/adversarial_data_set_fool_detectors.zip /content/
!unzip adversarial_data_set_fool_detectors.zip -d /content/

Archive:  adversarial_data_set_fool_detectors.zip
   creating: /content/content/adversarial_dataset_fool_detectors/
   creating: /content/content/adversarial_dataset_fool_detectors/labels/
  inflating: /content/content/adversarial_dataset_fool_detectors/labels/road466_png.rf.e1995d1db602b4fe4f26c9a2f882bf06.txt  
  inflating: /content/content/adversarial_dataset_fool_detectors/labels/road654_png.rf.46b39454d20cf1f488267148cc1ee23c.txt  
  inflating: /content/content/adversarial_dataset_fool_detectors/labels/road314_png.rf.b9b5f3356645589e1d35652eaf64b5bf.txt  
  inflating: /content/content/adversarial_dataset_fool_detectors/labels/00005_00016_00001_png.rf.8ed5c07e98a19127d55583cbd0ebabfb.txt  
  inflating: /content/content/adversarial_dataset_fool_detectors/labels/00002_00032_00021_png.rf.da1c231b696e926dff55375e5718a461.txt  
  inflating: /content/content/adversarial_dataset_fool_detectors/labels/road108_png.rf.3ccb5b5dbd4fe2ff044ce6f2aed3cf40.txt  
  inflating: /content/content/adver

In [34]:
# Evaluate fool detectors (didn't do adversarial training with it )
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1831.1±602.7 MB/s, size: 556.6 KB)


val: Scanning /content/content/adversarial_dataset_fool_detectors/labels... 638 images, 1 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<00:00, 938.65it/s]

val: New cache created: /content/content/adversarial_dataset_fool_detectors/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:16<00:00,  1.25it/s]


                   all        638        770      0.897      0.861      0.918      0.802
           Green Light         77        110      0.947      0.654      0.843      0.567
             Red Light         71         94       0.84      0.617      0.767        0.5
        Speed Limit 10          2          3      0.671      0.667      0.777      0.699
       Speed Limit 100         45         46      0.851      0.891      0.944      0.834
       Speed Limit 110         21         21      0.905      0.904      0.921      0.822
       Speed Limit 120         40         44      0.851      0.955      0.967      0.871
        Speed Limit 20         46         46      0.916      0.949      0.972      0.893
        Speed Limit 30         60         60      0.905       0.85      0.921      0.868
        Speed Limit 40         51         53      0.981      0.966       0.99      0.864
        Speed Limit 50         47         50      0.901      0.912      0.943      0.844
        Speed Limit 6

In [35]:
!rm -rf /content/content

In [36]:
!cp /content/drive/MyDrive/yolo/adversarial_data_set_dis_dag.zip /content/
!unzip adversarial_data_set_dis_dag.zip -d /content/

Archive:  adversarial_data_set_dis_dag.zip
   creating: /content/adversarial_data_set_dis_dag/
   creating: /content/adversarial_data_set_dis_dag/content/
   creating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/
  inflating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/labels.cache  
   creating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/images/
  inflating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/images/000359_jpg.rf.786658c4016fb105872988bc5cc4a3b3.png  
  inflating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/images/000099_jpg.rf.4f6c946e5e6bada45a6ed26cde81e0c7.png  
  inflating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/images/00014_00008_00013_png.rf.305dac45aa2ed53fd76943f24ae668c2.png  
  inflating: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/images/00014_00004_00010_png.rf.66187010c62b6f44f2dec0d41a63b804.png  
  infl

In [37]:
!cp -r test/labels/ adversarial_data_set_dis_dag/content/adversarial_dataset/

In [38]:
# Evaluate dis dag (didn't do adversarial training with it )
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1520.3±154.1 MB/s, size: 476.9 KB)


val: Scanning /content/adversarial_data_set_dis_dag/content/adversarial_dataset/labels... 113 images, 0 backgrounds, 0 corrupt: 100%|██████████| 113/113 [00:00<00:00, 961.88it/s]

val: New cache created: /content/adversarial_data_set_dis_dag/content/adversarial_dataset/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.21s/it]


                   all        113        139      0.873      0.921      0.943      0.841
           Green Light         14         23       0.95      0.826      0.902      0.577
             Red Light          9         12      0.562       0.75      0.751      0.536
       Speed Limit 100          5          6      0.857          1      0.995       0.88
       Speed Limit 110          3          3       0.75          1      0.995      0.937
       Speed Limit 120          4          8      0.636      0.875      0.915      0.785
        Speed Limit 20         11         11      0.917          1      0.988      0.907
        Speed Limit 30          8          8      0.857       0.75      0.856      0.843
        Speed Limit 40          5          5          1          1      0.995      0.895
        Speed Limit 50          8          8      0.889          1      0.982      0.919
        Speed Limit 60          7          7      0.875          1      0.995      0.924
        Speed Limit 7

In [39]:
!rm -rf /content/adversarial_data_set_dis_dag/

In [40]:
!cp /content/drive/MyDrive/yolo/adversarial_data_set_pgd_02.zip /content/
!unzip adversarial_data_set_pgd_02.zip -d /content/
!mv /content/content/adversarial_dataset/ /content/adversarial_dataset

Archive:  adversarial_data_set_pgd_02.zip
   creating: /content/content/adversarial_dataset/
  inflating: /content/content/adversarial_dataset/labels.cache  
   creating: /content/content/adversarial_dataset/labels/
  inflating: /content/content/adversarial_dataset/labels/00004_00060_00021_png.rf.a81a5ba2fb654d944063ddd496a46662.txt  
  inflating: /content/content/adversarial_dataset/labels/road446_png.rf.c209c8b5929313d1258667af1a6a37ae.txt  
  inflating: /content/content/adversarial_dataset/labels/00001_00028_00017_png.rf.98d01270a314b8cc77f6ad92f979b47f.txt  
  inflating: /content/content/adversarial_dataset/labels/001261_jpg.rf.7aab90697a4784411cbc2785e626e004.txt  
  inflating: /content/content/adversarial_dataset/labels/road59_png.rf.40b97d5dddbee9679979779c88bb7348.txt  
  inflating: /content/content/adversarial_dataset/labels/000811_jpg.rf.f5bb7753902ae5b5fd8f5e7f601da271.txt  
  inflating: /content/content/adversarial_dataset/labels/road692_png.rf.ab199ab022b948479bfba7b3c8fd6

In [41]:
# Evaluate PGD 0.02 (didn't do adversarial training with it )
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2351.2±965.1 MB/s, size: 703.9 KB)


val: Scanning /content/adversarial_dataset/labels.cache... 629 images, 1 backgrounds, 0 corrupt: 100%|██████████| 629/629 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:14<00:00,  1.35it/s]


                   all        629        760      0.933      0.883      0.937      0.816
           Green Light         74        107      0.908      0.738      0.888      0.583
             Red Light         70         93      0.885      0.662      0.826      0.537
        Speed Limit 10          2          3      0.751      0.667      0.732      0.659
       Speed Limit 100         44         45      0.934      0.947      0.988      0.874
       Speed Limit 110         19         19      0.998      0.947      0.972      0.849
       Speed Limit 120         39         43      0.892      0.964      0.982      0.887
        Speed Limit 20         46         46      0.957      0.976      0.987      0.912
        Speed Limit 30         60         60      0.952      0.867      0.936      0.886
        Speed Limit 40         51         53      0.968      0.962      0.979      0.858
        Speed Limit 50         47         50      0.948        0.9      0.935      0.864
        Speed Limit 6

In [42]:
!rm -rf /content/adversarial_dataset/
!cp /content/drive/MyDrive/yolo/adversarial_data_set_square4.zip /content/
!unzip adversarial_data_set_square4.zip -d /content/
!mv /content/content/adversarial_dataset/ /content/adversarial_dataset

Archive:  adversarial_data_set_square4.zip
   creating: /content/content/adversarial_dataset/
  inflating: /content/content/adversarial_dataset/labels.cache  
   creating: /content/content/adversarial_dataset/labels/
  inflating: /content/content/adversarial_dataset/labels/00004_00060_00021_png.rf.a81a5ba2fb654d944063ddd496a46662.txt  
  inflating: /content/content/adversarial_dataset/labels/road446_png.rf.c209c8b5929313d1258667af1a6a37ae.txt  
  inflating: /content/content/adversarial_dataset/labels/00001_00028_00017_png.rf.98d01270a314b8cc77f6ad92f979b47f.txt  
  inflating: /content/content/adversarial_dataset/labels/001261_jpg.rf.7aab90697a4784411cbc2785e626e004.txt  
  inflating: /content/content/adversarial_dataset/labels/road59_png.rf.40b97d5dddbee9679979779c88bb7348.txt  
  inflating: /content/content/adversarial_dataset/labels/000811_jpg.rf.f5bb7753902ae5b5fd8f5e7f601da271.txt  
  inflating: /content/content/adversarial_dataset/labels/road692_png.rf.ab199ab022b948479bfba7b3c8fd

In [43]:
# Evaluate square attack (black box)
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2076.3±1003.7 MB/s, size: 865.2 KB)


val: Scanning /content/adversarial_dataset/labels.cache... 618 images, 1 backgrounds, 0 corrupt: 100%|██████████| 618/618 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:11<00:00,  1.68it/s]


                   all        618        750      0.897      0.849      0.919      0.802
           Green Light         77        110      0.957        0.7      0.871      0.573
             Red Light         70         93      0.803      0.699      0.783      0.511
        Speed Limit 10          2          3      0.555      0.667      0.749      0.749
       Speed Limit 100         44         45      0.825      0.911      0.948      0.836
       Speed Limit 110         21         21          1      0.706       0.89      0.788
       Speed Limit 120         38         42      0.889      0.951      0.962      0.809
        Speed Limit 20         43         43      0.913       0.98      0.989      0.912
        Speed Limit 30         59         59      0.912      0.864      0.931      0.878
        Speed Limit 40         50         52      0.957      0.942      0.968      0.845
        Speed Limit 50         46         49      0.975      0.776      0.923      0.842
        Speed Limit 6

In [44]:
!rm -rf /content/adversarial_dataset/

In [45]:
!cp /content/drive/MyDrive/yolo/adversarial_test_images_hop.zip /content/
!unzip adversarial_test_images_hop.zip -d /content/
!mv /content/content/adversarial_dataset/ /content/adversarial_dataset

Archive:  adversarial_test_images_hop.zip
   creating: /content/content/adversarial_dataset/
   creating: /content/content/adversarial_dataset/labels/
  inflating: /content/content/adversarial_dataset/labels/08724eba-901b-4e19-883d-1cc236532391_jpg.rf.696238ad3396636709d743d91cf0a327.txt  
  inflating: /content/content/adversarial_dataset/labels/FisheyeCamera_1_00142_png.rf.b1d847a33308f0f39ada9349c4eff73c.txt  
  inflating: /content/content/adversarial_dataset/labels/b413caec-028d-48e6-922b-fc63db1b7ebb_jpg.rf.b98927268111bd394af386da9752473a.txt  
  inflating: /content/content/adversarial_dataset/labels/000599_jpg.rf.5fc75a3f1638e89f7c475fdd044ae210.txt  
  inflating: /content/content/adversarial_dataset/labels/FisheyeCamera_1_00100_png.rf.cbcaae5b7e34bc08d497476d0f770f12.txt  
  inflating: /content/content/adversarial_dataset/labels/001484_jpg.rf.b25e3dba3c08d7786aa499bebdb94e52.txt  
  inflating: /content/content/adversarial_dataset/labels/000380_jpg.rf.c4ff7dc3694d1a85c9337d273df0

In [46]:
# Evaluate hop skip attack (black box)
results = model.val(
    data="/content/adv.yaml",
    split="test",  # Not actually used but required by API
    imgsz=640,
    batch=32,
    conf = 0.25
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1503.9±117.8 MB/s, size: 783.0 KB)


val: Scanning /content/adversarial_dataset/labels.cache... 638 images, 1 backgrounds, 0 corrupt: 100%|██████████| 638/638 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:12<00:00,  1.66it/s]


                   all        638        770      0.921      0.862      0.931      0.809
           Green Light         77        110      0.937      0.673      0.875      0.562
             Red Light         71         94      0.879      0.698       0.82      0.518
        Speed Limit 10          2          3      0.735      0.667      0.749      0.674
       Speed Limit 100         45         46       0.89      0.913      0.963      0.859
       Speed Limit 110         21         21      0.931      0.857      0.938      0.824
       Speed Limit 120         40         44      0.914      0.966      0.984      0.879
        Speed Limit 20         46         46      0.957      0.975      0.984      0.908
        Speed Limit 30         60         60      0.929      0.869      0.948      0.889
        Speed Limit 40         51         53      0.924      0.962      0.979       0.86
        Speed Limit 50         47         50      0.977      0.838      0.936      0.854
        Speed Limit 6

### mAP@0.5 results after multi-attack adversarial training with 8252 adversarial images (2,000 PGD, 3,000 FGSM, 500 Targeted DAG, 1,500 UAP, 1,500 fool detectors) on top of your 10,590 clean images.
| Attack Technique        | Clean mAP@0.5 | Adv mAP@0.5 (Pre-adversarial training) | Adv mAP@0.5 (Post-adversarial Training) | Drop (Pre) | Drop (Post) | Notes |
|-------------------------|---------------|----------------------------|-----------------------------|------------|-------------|-------|
| Clean                   | 0.927         | -                          | 0.941                       | -          | -           | Slight clean mAP improvement (+0.012) |
| FGSM                    | 0.927         | 0.312                      | 0.874                       | -0.615     | -0.065      | Significant gain (+0.552), but still weakest |
| PGD (ε=0.1, α=0.025)    | 0.927         | 0.215                      | 0.92                       | -0.712     | -0.007      | Strong improvement (+0.700), very robust |
| Targeted DAG            | 0.927         | 0.0947                     | 0.927                       | -0.8323    | -0.00      | Huge gain (+0.8203), matches PGD robustness |
| UAP                     | 0.927         | 0.443                      | 0.917                       | -0.484     | -0.022      | Large gain (+0.474), near-clean performance |
| Fool Detectors          | 0.927         | 0.155                      | 0.918                       | -0.772     | -0.009      | Good gain (+0.742), but not trained on |
| Disappearance DAG       | 0.927         | 0.547                      | 0.943                       | -0.388     | +0.016      | Excellent (+0.390), near-clean, not trained on |
| PGD (ε=0.02, α=0.005)   | 0.927         | 0.617                      | 0.938                       | -0.310     | -0.001      | Near-perfect (+0.321), not trained on |
| square attack (black box)   | 0.927         | 0.59                      | 0.919                       | -0.337     | -0.008      | Near-perfect , not trained on and black box|
| hop skip jump (black box)   | 0.927         | 0.638                    | 0.931                       | -0.289     | +0.004     | perfect , not trained on and black box|

### The table compares performance with **8,252 adversarial images** (2,000 PGD, 3,000 FGSM, 500 Targeted DAG, 1,500 UAP, 1,500 Fool Detectors) and **5,500 adversarial images** (2,000 PGD, 1,500 FGSM, 500 Targeted DAG, 1,500 UAP) on top of 10,590 clean images and preprocessing defense

| Attack Technique              | Clean mAP@0.5 | Adv mAP@0.5 (Pre) | Adv mAP@0.5 (Preproc) | Adv mAP@0.5 (Post, 8,252) | Adv mAP@0.5 (Post, 5,500) | Drop (Pre) | Drop (Post, 8,252) | Drop (Post, 5,500) | Notes                                                        |
| ----------------------------- | ------------- | ----------------- | ---------------------- | ------------------------- | ------------------------- | ---------- | ------------------ | ------------------ | ------------------------------------------------------------ |
| **Clean**                     | 0.927         | -                 | -                      | 0.941                     | 0.939                     | -          | -                  | -                  | 8,252: +0.014 clean mAP; 5,500: +0.012. Larger dataset slightly better. |
| **FGSM**                      | 0.927         | 0.312             | 0.690                  | 0.874                     | 0.864                     | -0.615     | -0.053             | -0.063             | Preprocessing boosts to +0.378; 8,252 post-training is stronger. |
| **PGD (ε=0.1, α=0.025)**      | 0.927         | 0.215             | 0.780                  | 0.920                     | 0.915                     | -0.712     | -0.007             | -0.012             | Preprocessing improves by +0.565; training still yields best recovery. |
| **Targeted DAG**              | 0.927         | 0.0947            | 0.800                  | 0.927                     | 0.915                     | -0.8323    | 0.000              | -0.012             | Preprocessing effective (+0.7053); training yields full recovery. |
| **UAP**                       | 0.927         | 0.443             | 0.766                  | 0.917                     | 0.917                     | -0.484     | -0.010             | -0.010             | Preprocessing helps (+0.323); post-training gives near-perfect results. |
| **Fool Detectors**            | 0.927         | 0.155             | 0.780                  | 0.918                     | 0.897                     | -0.772     | -0.009             | -0.030             | Preprocessing +0.625 boost; post-training even better for 8,252. |
| **Disappearance DAG**         | 0.927         | 0.547             | -                      | 0.943                     | 0.937                     | -0.380     | +0.016             | +0.010             | No preprocessing result; post-training exceeds clean mAP. |
| **PGD (ε=0.02, α=0.005)**     | 0.927         | 0.617             | -                      | 0.938                     | 0.938                     | -0.310     | +0.011             | +0.011             | No preprocessing result; training already excellent. |
| **Square Attack (Black-box)** | 0.927         | 0.590             | 0.730                  | 0.919                     | 0.919                     | -0.337     | -0.008             | -0.008             | Preprocessing +0.140; post-training stronger, black-box not trained on. |
| **Hop Skip Jump (Black-box)** | 0.927         | 0.638             | 0.810                  | 0.931                     | 0.929                     | -0.289     | +0.004             | +0.002             | Preprocessing helps (+0.172); post-training gives near-recovery. |

